# Phase 2 · 19단원 — ML 시스템 설계

> 모델은 시스템의 한 부분입니다.
> 설계가 **무엇을 예측할지, 누가 그걸 보고 뭘 할지, 얼마가 들지, 망가지면 어떻게 할지**를 정합니다.

- 원문: <https://aiengineeringfromscratch.com/lesson?path=phases/02-ml-fundamentals/19-ml-system-design>
- 예상 시간: 110분
- 먼저 보면 좋은 것: 9단원(모델 평가), 13단원(파이프라인)
- 표시 규칙: 📗 는 원문에 있는 내용, ✨ 는 제가 넣은 예시·실험, ⚠️ 는 원문과 측정이 달랐던 곳입니다.

---

## 이 단원이 푸는 문제

어떤 회사가 **"이탈 예측 모델 하나 만들어 주세요"** 라고 합니다.

데이터 과학자가 그라디언트 부스팅으로 학습시켜서 **AUC 0.91** 을 냅니다. 좋은 숫자죠.

**석 달 뒤, 아무도 그 모델을 안 씁니다.**

왜일까요? 📗 원문이 드는 이유 넷은 **전부 알고리즘과 무관합니다.**

1. 점수가 나오면 **무슨 행동을 할지** 아무도 안 정했습니다
2. 점수는 **주 1회** 나오는데 고객 응대팀은 **매일** 전화를 겁니다
3. 레이블을 해지 테이블에서 가져왔는데 거기 **해지 사유**도 같이 있었습니다. 특징 하나에 답이 들어간 거죠
4. 지원팀이 이미 쓰던 규칙("2주 이상 접속 안 하면 전화")과 **비교를 안 했습니다**

넷 다 **학습을 시작하기도 전에** 생긴 문제입니다. 모델 문제가 아니라 **설계 문제**예요.

## 그래서 이 단원이 만드는 것

설계 문서가 물어봐야 할 질문들을 하나씩 다루고, 그걸 **세 가지 도구**로 만듭니다.

1. **서빙 비용 계산기** — 배치로 할지 온라인으로 할지, 얼마가 드는지
2. **기준선 비교기** — 모델이 정말 기존 규칙을 이기는지
3. **설계 문서 채점기** — 빠진 항목을 찾아내는 체크리스트

전부 표준 라이브러리만 씁니다. 📗 원문 코드도 그렇고요.

---

## 이걸 배우면 뭘 할 수 있나

> **"AI로 뭐 좀 해 봅시다."**

이 말을 들었을 때 **되물을 질문 목록**이 생깁니다.

| 질문 | 어디서 | 안 물으면 생기는 일 |
| --- | --- | --- |
| 뭘 예측하고, 그걸 보고 **누가 뭘 하나요?** | 1절 | 점수만 쌓이고 아무도 안 씁니다 |
| 지금 쓰는 방법보다 **얼마나 나아야** 하나요? | 3절 | 규칙보다 못한 모델을 배포합니다 |
| **언제까지** 나와야 하고 **얼마**까지 쓸 수 있나요? | 4·5절 | 못 지킬 설계를 골라 놓고 나중에 갈아엎습니다 |
| **망가지면** 뭘 하나요? | 7절 | 장애 한가운데서 즉흥으로 정합니다 |

그리고 마지막으로, **모델이 자기 미래 데이터를 오염시키는 경우**(6절)를 알게 됩니다.
이건 측정 결과가 교과서와 좀 다릅니다.

---

## 이 단원의 줄거리

지금까지 18단원 동안 **모델 안쪽**을 다뤘습니다. 어떻게 학습하고, 어떻게 재고, 어떤 특징을 쓰는지요.

19단원부터는 **모델 바깥**입니다. 모델을 둘러싼 시스템을 설계합니다.

**먼저 말을 정확하게 바꿉니다.** 1절입니다.
"이탈을 줄이자"는 목표고, "이 사용자가 30일 안에 해지할 확률"은 과제입니다. 둘은 다릅니다.
그리고 **그 숫자를 보고 누가 뭘 할지**까지 정해야 비로소 과제가 됩니다.

**그다음 아예 ML을 쓰지 말아야 할 경우를 봅니다.** 2절이에요.
구글의 첫 번째 규칙이 "ML 없이 출시하는 걸 두려워하지 말라" 입니다.

**3절부터 숫자 이야기입니다.** 모델 점수 하나만 보면 좋은지 나쁜지 알 수 없습니다.
비교 대상이 필요하죠. 가장 어려운 상대는 **기존에 쓰던 규칙**입니다. 직접 붙여 봅니다.

**4·5절은 돈과 시간입니다.** 지연·처리량·비용 예산을 정하고,
배치로 돌릴지 온라인으로 돌릴지를 **가격을 매겨서** 고릅니다.
여기서 ⚠️ 원문 설명과 다른 결과가 하나 나옵니다.

**6절은 배포입니다.** 섀도 → 카나리 → A/B 순서와 되돌리는 법.

**7절이 이 단원에서 가장 안 알려진 함정입니다.** 모델이 내린 결정이
다음 모델의 학습 데이터를 바꿔 버리는 경우요. 측정해 보면 교과서 설명과 좀 다릅니다.

**8절은 망가질 때 뭘 할지**를 미리 정하는 절이고,
**9절**에서 지금까지를 설계 문서 한 장으로 묶어 **채점기**를 만듭니다.

> 한 줄로 줄이면 이렇습니다.
> **말을 정확히 하고 → 기준선과 비교하고 → 예산 안에서 서빙을 고르고
> → 안전하게 배포하고 → 함정(피드백 루프)을 알고 → 실패 계획을 세워 → 문서로 묶는다.**

<details>
<summary>절 목록 (펼쳐 보기)</summary>

| 절 | 내용 | 출처 |
| --- | --- | --- |
| 1 | 목표를 ML 과제로 바꾸는 네 가지 질문 | 📗 원문 |
| 2 | 언제 ML을 쓰지 말아야 하나 | 📗 원문 |
| 3 | 기준선 — 모델이 정말 이기나 | 📗 원문 + ✨ 측정 |
| 4 | 지연·처리량·비용 예산 | 📗 원문 |
| 5 | ⚠️ 배치 vs 온라인 — 직접 가격을 매겨 봅니다 | 📗 원문 + ✨ 측정 |
| 6 | 배포 패턴: 섀도 · 카나리 · A/B | 📗 원문 |
| 7 | ⚠️ 피드백 루프 — 모델이 자기 데이터를 바꿀 때 | 📗 원문 + ✨ 반증 |
| 8 | 실패 모드와 안전 기본값 | 📗 원문 |
| 9 | 설계 문서 채점기 | 📗 원문 |
| 10 | 정리 · Ship It · 용어 · 퀴즈 | 📗 원문 |

</details>

---

## 0. 준비

In [1]:
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.font_manager import findfont, FontProperties

warnings.filterwarnings("ignore")

후보들 = ["AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic"]
찾은것 = None
for 후보 in 후보들:
    try:
        if 후보.lower().replace(" ", "") in findfont(FontProperties(family=후보)).lower().replace(" ", ""):
            찾은것 = 후보
            break
    except Exception:
        pass
matplotlib.rcParams["font.family"] = ["DejaVu Sans"] + ([찾은것] if 찾은것 else [])
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["figure.dpi"] = 110

print("준비 끝!  한글 폰트:", 찾은것 or "(없음)")

준비 끝!  한글 폰트: AppleGothic


---

## 1. 목표를 ML 과제로 바꾸기

<sub>📗 원문 "From a business goal to an ML task"</sub>

"이탈을 줄이자"는 **목표**입니다. 과제가 아니에요. 모델에게 시킬 수가 없습니다.

목표를 과제로 바꾸려면 📗 원문의 **네 가지 질문**에 답해야 합니다.

1. 정확히 **무엇을**, **어느 대상**(사용자·결제·매장·상품)에 대해 예측하나?
2. 그 예측을 보고 **어떤 결정**을 하며, **누가/무엇이** 그 결정을 하나?
3. 예측이 **언제** 필요하고, 그 시점에 **어떤 데이터가 존재**하나?
4. 결정이 **나아졌다는 걸 어떻게 알** 건가?

### 📗 네 가지 예

| 목표 | 예측 | 그걸 쓰는 결정 | ML 과제 |
| --- | --- | --- | --- |
| 이탈 줄이기 | 30일 안에 해지할 확률 | 매일 밤 상위 5%에게 혜택 발송 | 이진 분류 |
| 사기 손실 줄이기 | 이 결제가 사기일 확률 | 100ms 안에 승인/검토/차단 | 이진 분류 |
| 재고 맞추기 | 다음 주 매장별 일 판매량 | 매장별 발주 수량 | 예측(회귀) |
| 관련 상품 보여주기 | (사용자, 상품) 관련도 점수 | 피드 정렬 순서 | 랭킹 |

**두 번째 열에 가장 공을 들여야 합니다.**

> "이탈 예측" ❌ — 기간도, 대상도, 행동도 안 정해졌습니다
> "이 사용자가 **30일 안에** 해지할 확률을 예측해서, **오늘 밤 작업이** 혜택 대상을 고른다" ✅

세 가지가 다 박혔습니다.

> **실험 1** · 말을 바꾸면 요구사항이 드러납니다
>
> **🎯 배울 것** — 같은 '이탈 예측'도 결정이 뭐냐에 따라 **전혀 다른 시스템**이 됩니다.
>
> **🔬 하는 일** — 같은 예측을 세 가지 다른 결정에 붙여 보고, 각각 필요한 신선도·지연·대상 수를 적어 봅니다.

In [2]:
# ✨ 추가 — 같은 예측, 다른 결정. 요구사항이 어떻게 갈리나
@dataclass(frozen=True)
class 과제:
    결정: str
    주기: str
    신선도시간: float
    지연예산ms: float
    대상수: int

과제들 = [
    과제("매일 밤 상위 5%에게 할인 쿠폰 발송", "하루 1회", 24.0, 60_000, 2_000_000),
    과제("고객이 해지 버튼을 누르면 즉시 만류 화면", "요청 즉시", 0.05, 200, 2_000_000),
    과제("주간 리포트에 위험 고객 목록 첨부", "주 1회", 168.0, 600_000, 2_000_000),
]

print("예측은 셋 다 똑같습니다: '이 사용자가 30일 안에 해지할 확률'")
print("바뀐 건 그 숫자를 보고 뭘 하느냐 뿐입니다.\n")
print(f"{'결정':<34}{'주기':<10}{'신선도':>9}{'지연 예산':>11}{'가능한 서빙':>14}")
print("-" * 80)
for t in 과제들:
    # 배치 작업 한 번이 1시간 걸린다고 보면
    배치가능 = t.신선도시간 > 1.0
    모드 = "배치 / 온라인" if 배치가능 else "온라인만"
    신선 = f"{t.신선도시간:.0f}시간" if t.신선도시간 >= 1 else f"{t.신선도시간*60:.0f}분"
    지연 = f"{t.지연예산ms/1000:.0f}초" if t.지연예산ms >= 1000 else f"{t.지연예산ms:.0f}ms"
    print(f"{t.결정:<33}{t.주기:<10}{신선:>9}{지연:>11}{모드:>14}")

print(f"\n=> 두 번째 줄을 보세요. 신선도가 3분이라 배치가 아예 불가능합니다.")
print(f"   배치 작업 한 번이 1시간 걸리는데 3분마다 새로 돌릴 수는 없으니까요.")
print(f"   5절에서 이걸 가격으로 계산합니다.")
print(f"\n   같은 예측인데 결정이 바뀌니 시스템이 통째로 달라집니다.")
print(f"   그래서 '무슨 결정을 하나'를 먼저 물어야 합니다.")

예측은 셋 다 똑같습니다: '이 사용자가 30일 안에 해지할 확률'
바뀐 건 그 숫자를 보고 뭘 하느냐 뿐입니다.

결정                                주기              신선도      지연 예산        가능한 서빙
--------------------------------------------------------------------------------
매일 밤 상위 5%에게 할인 쿠폰 발송            하루 1회          24시간        60초      배치 / 온라인
고객이 해지 버튼을 누르면 즉시 만류 화면          요청 즉시            3분      200ms          온라인만
주간 리포트에 위험 고객 목록 첨부              주 1회          168시간       600초      배치 / 온라인

=> 두 번째 줄을 보세요. 신선도가 3분이라 배치가 아예 불가능합니다.
   배치 작업 한 번이 1시간 걸리는데 3분마다 새로 돌릴 수는 없으니까요.
   5절에서 이걸 가격으로 계산합니다.

   같은 예측인데 결정이 바뀌니 시스템이 통째로 달라집니다.
   그래서 '무슨 결정을 하나'를 먼저 물어야 합니다.


> **👀 주목할 곳** — `가능한 서빙` 열. 두 번째 줄만 '온라인만' 입니다.
>
> **✅ 알게 된 사실** — 예측 내용은 셋 다 같은데 **결정이 신선도 요구를 정하고, 신선도가 서빙 방식을 정합니다.**
그래서 '무슨 결정을 하나'를 안 물으면 설계를 시작할 수가 없습니다.

> **🧩 이 절의 결론**
>
> 목표와 과제는 다릅니다. 과제가 되려면 **대상·기간·결정**이 박혀야 합니다.
> 특히 **결정**이 나머지를 끌고 갑니다. 결정이 신선도를, 신선도가 서빙 방식을 정합니다.
>
> **➡️ 다음**
>
> 그런데 그 전에 물어볼 게 하나 더 있습니다.
> **애초에 ML이 필요한가?** 2절입니다.

---

## 2. 언제 ML을 쓰지 말아야 하나

<sub>📗 원문 "When not to use ML"</sub>

> 📗 구글 *Rules of Machine Learning* 의 **1번 규칙**: 머신러닝 없이 제품을 내는 걸 두려워하지 마세요.

ML은 데이터가 있어야 하고, 간단한 휴리스틱이 꽤 멀리까지 데려다줍니다.

### 📗 ML을 쓰면 안 되는 여섯 경우

| 상황 | 왜 |
| --- | --- |
| 짧은 규칙으로 목표가 달성되고 그게 계속 맞을 때 | 규칙이 **더 싸고, 테스트하기 쉽고, 설명하기 쉽습니다** |
| 레이블이 없고 모을 방법도 마땅찮을 때 | 학습할 게 없습니다 |
| 아무도 그 예측을 보고 행동하지 않을 때 | 더 좋은 예측이 **아무것도 바꾸지 않습니다** |
| 예측 시점에 존재하는 데이터에 신호가 없을 때 | 못 맞힙니다 |
| 한 번의 오류도 용납 안 되는데 사람이 검토할 수 없을 때 | 위험합니다 |
| 만들고 굴리는 비용이 **더 나은 결정의 가치**보다 클 때 | 손해입니다 |

### 📗 반대로 ML이 유리해지는 지점

> 규칙이 **길어지고 깨지기 쉬워질 때**입니다.
> 같은 문서의 3번 규칙은 복잡한 휴리스틱보다 ML을 택하라고 합니다.
> 학습된 모델이 **고치고 유지하기 더 쉽기** 때문입니다.

세 번째 줄을 특히 기억해 두세요. 도입부의 실패 사례 1번이 정확히 그거였습니다.
점수는 나왔는데 **아무도 그걸로 뭘 할지 몰랐던** 경우요.

> **🧩 이 절의 결론**
>
> 규칙이 **짧고 안정적이면** 규칙이 낫습니다. ML은 규칙이 길어지고 깨질 때 값을 합니다.
> 그리고 **아무도 행동하지 않을 예측**은 아무리 정확해도 소용없습니다.
>
> **➡️ 다음**
>
> 그럼 '규칙보다 나은지'를 어떻게 확인할까요? **재 보면 됩니다.** 3절입니다.

---

## 3. 기준선 — 모델이 정말 이기나

<sub>📗 원문 "Three layers of metrics" + "Baselines"</sub>

### 📗 먼저 지표를 세 층으로 나눕니다

| 층 | 언제 잴 수 있나 | 이탈 예시 |
| --- | --- | --- |
| **오프라인** | 출시 전, 과거 데이터로 | 혜택 임곗값에서의 F1, 상위 5% 정밀도 |
| **온라인** | 출시 중, 실제 트래픽에서 | 혜택 수락률, 점수가 매겨진 사용자 비율 |
| **비즈니스** | 출시 몇 주 뒤, 실험으로 | 처치군 vs 대조군의 30일 이탈률 |
| **가드레일** | 항상 | 구한 고객당 혜택 비용, 수신거부율, p99 지연 |

> 📗 원문: 세 층은 **자주 서로 다른 말을 합니다.**
> 오프라인 지표는 **내 모델이 없던 과거**를 채점합니다.
> 온라인 지표는 **사람들이 그 결정에 어떻게 반응하느냐**에 달렸고요.
> 비즈니스 지표는 느리게 움직이고 회사의 다른 모든 일의 잡음을 탑니다.

**결정에 맞는 오프라인 지표를 고르세요.** 상위 5%에게 연락한다면 **상위 5% 정밀도**를 봐야 합니다.
AUC는 모든 임곗값을 평균 내는데, 그중 **쓰지도 않을 임곗값**이 잔뜩 들어 있습니다.

### 📗 그리고 기준선과 비교합니다

> F1 0.56은 최선의 대안이 0.30이면 **훌륭하고**, 규칙이 0.60이면 **형편없습니다.**

| 기준선 | 뜻 |
| --- | --- |
| **자명한 기준선** | 늘 다수 클래스, 또는 늘 평균값 |
| **휴리스틱 기준선** | 도메인 규칙, 또는 특징 하나짜리 임곗값 |
| **현재 시스템** | 지금 그 결정을 하고 있는 것. 사람이 엑셀로 하고 있어도 |

**리프트**는 모델 점수에서 **가장 센 기준선**을 뺀 값입니다. 가장 약한 게 아니라요.
그리고 **결과를 보기 전에** 최소 리프트를 정해 두세요.

> **실험 3** · 모델 vs 기준선 세 개
>
> **🎯 배울 것** — 모델 점수는 혼자서는 아무 뜻이 없습니다. **가장 센 기준선**과 비교해야 합니다.
>
> **🔬 하는 일** — 이탈 데이터를 만들어 로지스틱 모델과 기준선 셋(다수 클래스·무작위·규칙)을 같은 F1으로 채점합니다.

In [3]:
# 📗 원문의 데이터 생성과 기준선들을 그대로 옮깁니다
특징들 = ["days_idle", "tickets", "tenure_months"]


def 이탈데이터(n, 시드, tickets계수=0.45, tenure계수=-0.035):
    rng = random.Random(시드)
    행들, 레이블 = [], []
    for _ in range(n):
        idle = rng.expovariate(1 / 10)
        tickets = rng.choice([0, 0, 0, 1, 1, 2, 3, 5])
        tenure = rng.uniform(1, 60)
        z = -2.6 + 0.12 * idle + tickets계수 * tickets + tenure계수 * tenure
        레이블.append(1 if rng.random() < 1 / (1 + math.exp(-z)) else 0)
        행들.append({"days_idle": idle, "tickets": tickets, "tenure_months": tenure})
    return 행들, 레이블


def F1(참, 예측):
    tp = sum(1 for t, p in zip(참, 예측) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(참, 예측) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(참, 예측) if t == 1 and p == 0)
    return 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)


def 다수클래스(y훈련, n):
    많은쪽 = max(sorted(set(y훈련)), key=y훈련.count)
    return [많은쪽] * n


def 기저비율(y훈련, n, 시드=0):
    비율 = sum(y훈련) / len(y훈련)
    rng = random.Random(시드)
    return [1 if rng.random() < 비율 else 0 for _ in range(n)]


def 규칙학습(행들, 레이블, 특징):
    # 특징 하나로 F1 이 가장 높은 임곗값을 찾습니다
    최적, 최고 = None, -1.0
    for 후보 in sorted(set(r[특징] for r in 행들)):
        점수 = F1(레이블, [1 if r[특징] > 후보 else 0 for r in 행들])
        if 점수 > 최고:
            최적, 최고 = 후보, 점수
    return 최적


class 로지스틱:
    def __init__(self, 반복=300, 학습률=0.5):
        self.반복, self.학습률 = 반복, 학습률

    def _벡터(self, r):
        return [(r[k] - self.평균[k]) / self.표준[k] for k in 특징들]

    def _시그모이드(self, x):
        z = sum(w * xj for w, xj in zip(self.w, x)) + self.b
        return 1 / (1 + math.exp(-max(-50.0, min(50.0, z))))

    def fit(self, 행들, 레이블):
        n = len(행들)
        self.평균 = {k: sum(r[k] for r in 행들) / n for k in 특징들}
        self.표준 = {k: (math.sqrt(sum((r[k] - self.평균[k]) ** 2 for r in 행들) / n) or 1.0)
                    for k in 특징들}
        벡터들 = [self._벡터(r) for r in 행들]
        self.w = [0.0] * len(특징들)
        self.b = 0.0
        for _ in range(self.반복):
            gw = [0.0] * len(특징들)
            gb = 0.0
            for x, y in zip(벡터들, 레이블):
                오차 = self._시그모이드(x) - y
                for j, xj in enumerate(x):
                    gw[j] += 오차 * xj
                gb += 오차
            self.w = [w - self.학습률 * g / n for w, g in zip(self.w, gw)]
            self.b -= self.학습률 * gb / n
        확률 = [self._시그모이드(x) for x in 벡터들]
        격자 = [t / 100 for t in range(5, 96, 5)]
        self.임곗값 = max(격자, key=lambda t: F1(레이블, [1 if p >= t else 0 for p in 확률]))
        return self

    def predict(self, 행들):
        return [1 if self._시그모이드(self._벡터(r)) >= self.임곗값 else 0 for r in 행들]


훈련, y훈련 = 이탈데이터(4000, 7)
테스트, y테스트 = 이탈데이터(2000, 8)
모델 = 로지스틱().fit(훈련, y훈련)
임곗값 = 규칙학습(훈련, y훈련, "days_idle")

점수표 = {
    "모델 (로지스틱)": F1(y테스트, 모델.predict(테스트)),
    f"규칙: days_idle > {임곗값:.1f}": F1(y테스트, [1 if r["days_idle"] > 임곗값 else 0 for r in 테스트]),
    "무작위 (기저 비율대로)": F1(y테스트, 기저비율(y훈련, len(테스트))),
    "다수 클래스": F1(y테스트, 다수클래스(y훈련, len(테스트))),
}

print(f"이탈 데이터: 훈련 {len(훈련)}건, 테스트 {len(테스트)}건, 실제 이탈률 {sum(y테스트)/len(y테스트):.1%}\n")
print(f"{'방법':<26}{'F1':>8}   종류")
print("-" * 52)
종류표 = ["학습한 모델", "휴리스틱 기준선", "자명한 기준선", "자명한 기준선"]
for (이름, 점), 종 in zip(점수표.items(), 종류표):
    print(f"{이름:<25}{점:>8.3f}   {종}")

기준선들 = {k: v for k, v in 점수표.items() if k != "모델 (로지스틱)"}
최강 = max(기준선들, key=기준선들.get)
리프트 = 점수표["모델 (로지스틱)"] - 기준선들[최강]
print(f"\n=> 가장 센 기준선은 {최강} 으로 {기준선들[최강]:.3f} 입니다.")
print(f"   모델의 리프트 = {점수표['모델 (로지스틱)']:.3f} - {기준선들[최강]:.3f} = {리프트:+.3f}")
print(f"\n   '다수 클래스'가 F1 0.000 인 걸 보세요. 이탈을 한 번도 예측 안 하니까요.")
print(f"   이 기준선과 비교하면 어떤 모델이든 이깁니다. 비교 대상이 될 수 없습니다.")
print(f"   진짜 상대는 규칙입니다.")

이탈 데이터: 훈련 4000건, 테스트 2000건, 실제 이탈률 20.3%

방법                              F1   종류
----------------------------------------------------
모델 (로지스틱)                   0.588   학습한 모델
규칙: days_idle > 15.9        0.529   휴리스틱 기준선
무작위 (기저 비율대로)               0.180   자명한 기준선
다수 클래스                      0.000   자명한 기준선

=> 가장 센 기준선은 규칙: days_idle > 15.9 으로 0.529 입니다.
   모델의 리프트 = 0.588 - 0.529 = +0.059

   '다수 클래스'가 F1 0.000 인 걸 보세요. 이탈을 한 번도 예측 안 하니까요.
   이 기준선과 비교하면 어떤 모델이든 이깁니다. 비교 대상이 될 수 없습니다.
   진짜 상대는 규칙입니다.


> **👀 주목할 곳** — `종류` 열을 보면서 **휴리스틱 기준선** 줄을 찾으세요. 자명한 기준선 둘보다 훨씬 셉니다.
>
> **✅ 알게 된 사실** — 다수 클래스는 F1 0.000 입니다. 이걸 이겼다고 자랑하면 안 됩니다.
**리프트는 가장 센 기준선 기준으로** 재야 하고, 여기서는 규칙 대비 +0.059 입니다.

> **실험 3-B** · 규칙이 모델을 이기는 경우
>
> **🎯 배울 것** — 모델이 늘 이기는 게 아닙니다. **규칙이 못 보는 특징이 약해지면** 따라잡힙니다.
>
> **🔬 하는 일** — 📗 연습문제 2번입니다. 규칙이 쓰는 `days_idle` 외의 특징(tickets·tenure) 영향을 줄여 가며 둘을 겨룹니다.

In [4]:
# 📗 연습문제 2번 — 규칙이 못 보는 특징을 약화시키면
print("규칙은 `days_idle` 하나만 봅니다. 나머지 둘은 못 보죠.")
print("그 둘의 영향을 줄여 가며 모델과 규칙을 겨뤄 봅니다.\n")
print(f"{'tickets 계수':>12}{'tenure 계수':>12}{'모델 F1':>10}{'규칙 F1':>10}{'리프트':>10}  승자")
print("-" * 68)
for t계수, te계수 in [(0.45, -0.035), (0.30, -0.035), (0.15, -0.035),
                   (0.10, -0.010), (0.00, 0.000)]:
    tr, ytr = 이탈데이터(4000, 7, t계수, te계수)
    te, yte = 이탈데이터(2000, 8, t계수, te계수)
    m = 로지스틱().fit(tr, ytr)
    임 = 규칙학습(tr, ytr, "days_idle")
    a = F1(yte, m.predict(te))
    b = F1(yte, [1 if r["days_idle"] > 임 else 0 for r in te])
    print(f"{t계수:>12.2f}{te계수:>12.3f}{a:>10.3f}{b:>10.3f}{a-b:>+10.3f}  "
          f"{'모델' if a > b else '규칙'}")

print(f"\n=> 맨 아래 두 줄에서 규칙이 이깁니다.")
print(f"   규칙이 못 보는 특징(tickets·tenure)의 영향이 사라지니,")
print(f"   모델도 `days_idle` 말고는 볼 게 없어져서 규칙과 같아집니다.")
print(f"\n   📗 원문: 'With weaker features the rule can win,")
print(f"   and then the rule is the right system to deploy.'")
print(f"   규칙이 이기면 규칙을 배포하는 게 맞습니다. 그게 더 싸고 설명하기 쉬우니까요.")

규칙은 `days_idle` 하나만 봅니다. 나머지 둘은 못 보죠.
그 둘의 영향을 줄여 가며 모델과 규칙을 겨뤄 봅니다.

  tickets 계수   tenure 계수     모델 F1     규칙 F1       리프트  승자
--------------------------------------------------------------------
        0.45      -0.035     0.588     0.529    +0.059  모델
        0.30      -0.035     0.562     0.529    +0.033  모델
        0.15      -0.035     0.531     0.514    +0.016  모델
        0.10      -0.010     0.553     0.555    -0.002  규칙
        0.00       0.000     0.550     0.551    -0.002  규칙

=> 맨 아래 두 줄에서 규칙이 이깁니다.
   규칙이 못 보는 특징(tickets·tenure)의 영향이 사라지니,
   모델도 `days_idle` 말고는 볼 게 없어져서 규칙과 같아집니다.

   📗 원문: 'With weaker features the rule can win,
   and then the rule is the right system to deploy.'
   규칙이 이기면 규칙을 배포하는 게 맞습니다. 그게 더 싸고 설명하기 쉬우니까요.


> **👀 주목할 곳** — `승자` 열이 '모델'에서 '규칙'으로 바뀌는 줄.
>
> **✅ 알게 된 사실** — 모델의 값어치는 **규칙이 못 보는 것을 볼 때** 생깁니다.
그게 없으면 리프트가 0 근처가 되고, 그때는 규칙을 배포하는 게 맞습니다. 더 싸고 설명하기 쉬우니까요.

> **🧩 이 절의 결론**
>
> 모델 점수는 혼자서는 뜻이 없습니다. **가장 센 기준선**과 비교해야 합니다.
> 그리고 그 상대는 보통 **기존 규칙**이지 '다수 클래스'가 아닙니다.
> 규칙이 이기면 규칙을 배포하는 게 맞습니다.
>
> **➡️ 다음**
>
> 모델이 규칙을 이긴다고 칩시다. 그럼 이제 **어떻게 서빙할지**를 정해야 합니다.
> 그 전에 예산부터 세웁니다. 4절입니다.

---

## 4. 지연 · 처리량 · 비용 예산

<sub>📗 원문 "Latency, throughput, and cost budgets"</sub>

서빙 설계를 묶는 숫자가 **셋**입니다.

| 예산 | 뜻 | 주의 |
| --- | --- | --- |
| **지연** | p99 지연 — 요청 100건 중 99건이 이 시간 안에 끝납니다 | 평균이 아니라 **p99** 를 보세요 |
| **최대 처리량** | 가장 바쁜 1분 기준 QPS | 일 평균이 아니라 **피크** 로 잡습니다 |
| **비용** | 하루 달러, 또는 예측 1000건당 달러 | 설계끼리 비교할 수 있게 **단위를 맞춥니다** |

### 📗 Little's 법칙이 앞 둘을 묶습니다

$$\text{동시에 처리 중인 요청 수} = \text{처리량} \times \text{지연}$$

초당 400건이 들어오고 한 건에 50ms 걸리면, 어느 순간에나 **20건이 처리 중**입니다.
서버를 몇 대 둘지, 커넥션 풀을 얼마로 잡을지가 여기서 나옵니다.

> **실험 4** · Little's 법칙
>
> **🎯 배울 것** — 처리량과 지연을 알면 **동시에 떠 있는 요청 수**가 바로 나옵니다.
>
> **🔬 하는 일** — 📗 원문의 예(400 QPS × 50ms)를 재현하고, 숫자를 바꿔 가며 동시 요청 수를 봅니다.

In [5]:
print("동시 처리 건수 = 초당 요청 수 × 한 건에 걸리는 시간\n")
# 5절 가격표와 똑같이, 복제 한 대가 200 QPS 를 감당한다고 둡니다
대당QPS = 200
print(f"{'상황':<26}{'QPS':>8}{'지연':>9}{'동시 처리':>11}{'필요 복제 수':>14}")
print("-" * 72)
for 라벨, qps, ms in [("📗 원문 예시", 400, 50),
                    ("야간 이탈 배치 조회", 40, 4),
                    ("피드 랭킹", 120, 35),
                    ("결제 사기 검사", 60, 35),
                    ("급증한 결제 트래픽", 2000, 35)]:
    동시 = qps * ms / 1000
    복제 = max(2, math.ceil(qps / 대당QPS))
    print(f"{라벨:<25}{qps:>8}{ms:>7}ms{동시:>11.1f}{복제:>13}대")

print(f"\n=> 📗 원문: '400 QPS, 50ms 면 어느 순간에나 20건이 처리 중' ✔")
print(f"\n   맨 아랫줄을 보세요. 트래픽이 2000 QPS 로 뛰면 동시 70건이고,")
print(f"   복제가 2대에서 10대로 늘어납니다. 지연은 그대로인데도요.")
print(f"   그래서 피크 기준으로 설계합니다. 평균으로 잡으면 가장 바쁠 때 무너집니다.")
print(f"\n   위 네 줄은 복제가 전부 최소치인 2대입니다. 트래픽이 작아서요.")
print(f"   이게 5절에서 온라인 비용이 고정비처럼 보이는 이유입니다.")

동시 처리 건수 = 초당 요청 수 × 한 건에 걸리는 시간

상황                             QPS       지연      동시 처리       필요 복제 수
------------------------------------------------------------------------
📗 원문 예시                       400     50ms       20.0            2대
야간 이탈 배치 조회                    40      4ms        0.2            2대
피드 랭킹                         120     35ms        4.2            2대
결제 사기 검사                       60     35ms        2.1            2대
급증한 결제 트래픽                   2000     35ms       70.0           10대

=> 📗 원문: '400 QPS, 50ms 면 어느 순간에나 20건이 처리 중' ✔

   맨 아랫줄을 보세요. 트래픽이 2000 QPS 로 뛰면 동시 70건이고,
   복제가 2대에서 10대로 늘어납니다. 지연은 그대로인데도요.
   그래서 피크 기준으로 설계합니다. 평균으로 잡으면 가장 바쁠 때 무너집니다.

   위 네 줄은 복제가 전부 최소치인 2대입니다. 트래픽이 작아서요.
   이게 5절에서 온라인 비용이 고정비처럼 보이는 이유입니다.


> **👀 주목할 곳** — `동시 처리` 열. 지연이 같아도 QPS가 오르면 같이 오릅니다.
>
> **✅ 알게 된 사실** — 서버 대수와 커넥션 풀 크기가 이 숫자에서 나옵니다.
**평균이 아니라 피크로** 잡아야 가장 바쁠 때 안 무너집니다.

> **🧩 이 절의 결론**
>
> 예산은 **지연 · 피크 처리량 · 비용** 셋입니다.
> Little's 법칙이 앞 둘을 묶어서 필요한 서버 규모를 알려 줍니다.
>
> **➡️ 다음**
>
> 예산이 정해졌으니 이제 **서빙 방식**을 고릅니다.
> 배치냐 온라인이냐. 5절에서 **가격을 매겨서** 고릅니다.

---

## 5. ⚠️ 배치 vs 온라인 — 가격을 매겨 봅니다

<sub>📗 원문 "Batch inference versus online inference"</sub>

예측을 내놓는 방법이 크게 둘입니다.

- **배치 추론** — 정해진 시각에 작업이 돌면서 **모든 대상의 점수를 미리 계산**해 테이블에 적어 둡니다.
  요청이 오면 그 값을 **읽기만** 합니다.
- **온라인 추론** — 요청이 올 때마다 **그 자리에서** 모델을 돌립니다.

| 성질 | 배치 | 온라인 |
| --- | --- | --- |
| 신선도 | 마지막 실행 시각 + 작업 시간만큼 낡음 | 요청 시점 데이터를 씀 |
| 요청 지연 | 키-값 조회 한 번, 몇 ms | 특징 조회 + 모델 시간 |
| 비용 동인 | 대상 수 × 하루 실행 횟수 | **피크에 맞춘 복제 수** |
| 새 대상 | 다음 실행까지 점수 없음 | 첫 요청에 바로 점수 |
| 망가질 때 | 옛 점수는 계속 읽힘 | 요청이 실패하거나 타임아웃 |

### 📗 원문의 가격표

실제 견적이 아니라 **연습용 반올림 숫자**입니다.

| 항목 | 값 |
| --- | --- |
| 배치 예측 100만 건당 | $0.40 |
| 배치 작업 1회 소요 | 1시간 |
| 키-값 조회 100만 건당 | $0.25 · 4ms |
| 복제 1대 시간당 | $0.50 · 200 QPS 감당 |
| 최소 복제 수 | 2대 (가용성) |
| 온라인 지연 | 35ms |

여기서 **중요한 제약**이 하나 나옵니다. 배치 작업이 1시간 걸리면
**신선도 1시간 미만은 어떤 가격을 줘도 불가능합니다.**

> **실험 5** · 배치와 온라인에 가격 매기기
>
> **🎯 배울 것** — 서빙 방식은 취향이 아니라 **계산 결과**로 고릅니다.
>
> **🔬 하는 일** — 📗 원문의 가격표로 배치·온라인 비용 함수를 만들고, 원문의 세 가지 작업에 적용합니다.

In [6]:
# 📗 원문의 CostModel 과 두 plan 함수를 그대로 옮깁니다
@dataclass(frozen=True)
class 작업부하:
    이름: str
    대상수: int
    일요청수: int
    피크QPS: float
    최대신선도시간: float
    지연예산ms: float


@dataclass(frozen=True)
class 가격표:
    배치_백만당: float = 0.40
    배치_작업시간: float = 1.0
    조회_백만당: float = 0.25
    조회_지연ms: float = 4.0
    복제_시간당: float = 0.50
    복제_QPS: float = 200.0
    최소복제: int = 2
    온라인_지연ms: float = 35.0


def 배치안(w, p):
    간격 = w.최대신선도시간 - p.배치_작업시간
    if 간격 <= 0:
        return {"가능": False, "이유": "신선도 한도가 배치 작업 1회보다 짧습니다"}
    회수 = 24.0 / 간격
    예측수 = w.대상수 * 회수
    돈 = 예측수 / 1e6 * p.배치_백만당 + w.일요청수 / 1e6 * p.조회_백만당
    return {"가능": p.조회_지연ms <= w.지연예산ms, "돈": 돈, "지연": p.조회_지연ms,
            "메모": f"하루 {회수:.1f}회 × 대상 {w.대상수:,}"}


def 온라인안(w, p):
    복제 = max(p.최소복제, math.ceil(w.피크QPS / p.복제_QPS))
    돈 = 복제 * p.복제_시간당 * 24.0
    return {"가능": p.온라인_지연ms <= w.지연예산ms, "돈": 돈, "지연": p.온라인_지연ms,
            "메모": f"복제 {복제}대 (피크 {w.피크QPS:.0f} QPS 기준)"}


가격 = 가격표()
# 📗 원문의 세 가지 작업부하를 숫자 그대로 씁니다
작업들 = [
    작업부하("야간 이탈 점수", 2_000_000, 300_000, 40, 24.0, 100),
    작업부하("피드 랭킹 (2시간 신선도)", 5_000_000, 2_000_000, 120, 2.0, 80),
    작업부하("결제 시 사기 검사", 10_000_000, 800_000, 60, 0.05, 50),
]

for w in 작업들:
    print(f"\n{w.이름}")
    후보 = []
    for 라벨, 안 in [("배치", 배치안(w, 가격)), ("온라인", 온라인안(w, 가격))]:
        if not 안["가능"]:
            이유 = 안.get("이유", f"지연 {안.get('지연', 0):.0f}ms > 예산 {w.지연예산ms:.0f}ms")
            print(f"   ✘ {라벨:<6}{'불가':>12}  {이유}")
        else:
            후보.append((라벨, 안))
            print(f"   ○ {라벨:<6}{'$'+format(안['돈'], '.2f')+'/일':>12}{안['지연']:>7.0f}ms  {안['메모']}")
    if 후보:
        고름 = min(후보, key=lambda t: t[1]["돈"])
        print(f"   => 선택: {고름[0]} (조건을 만족하는 것 중 가장 쌈)")
    else:
        print(f"   => ⚠️ 가능한 안이 없습니다. 예산을 바꿔야 합니다")

print(f"\n\n📗 원문 출력과 대조")
print(f"   야간 이탈: 배치 $0.91/일, 온라인 $24.00/일 -> 배치 ✔")
print(f"   피드 랭킹: 배치 $48.50/일, 온라인 $24.00/일 -> 온라인 ✔")


야간 이탈 점수
   ○ 배치         $0.91/일      4ms  하루 1.0회 × 대상 2,000,000
   ○ 온라인       $24.00/일     35ms  복제 2대 (피크 40 QPS 기준)
   => 선택: 배치 (조건을 만족하는 것 중 가장 쌈)

피드 랭킹 (2시간 신선도)
   ○ 배치        $48.50/일      4ms  하루 24.0회 × 대상 5,000,000
   ○ 온라인       $24.00/일     35ms  복제 2대 (피크 120 QPS 기준)
   => 선택: 온라인 (조건을 만족하는 것 중 가장 쌈)

결제 시 사기 검사
   ✘ 배치              불가  신선도 한도가 배치 작업 1회보다 짧습니다
   ○ 온라인       $24.00/일     35ms  복제 2대 (피크 60 QPS 기준)
   => 선택: 온라인 (조건을 만족하는 것 중 가장 쌈)


📗 원문 출력과 대조
   야간 이탈: 배치 $0.91/일, 온라인 $24.00/일 -> 배치 ✔
   피드 랭킹: 배치 $48.50/일, 온라인 $24.00/일 -> 온라인 ✔


> **👀 주목할 곳** — 세 번째 작업의 배치 줄. `불가` 입니다.
>
> **✅ 알게 된 사실** — 신선도 한도가 3분(0.05시간)인데 배치 작업 1회가 1시간 걸립니다.
**가격 문제가 아니라 구조적으로 불가능**합니다. 1절에서 예고한 그 줄이에요.

> **실험 5-B** · ⚠️ 뭐가 정말 비용을 가르나
>
> **🎯 배울 것** — 원문은 '배치 비용은 대상 수에 비례'라고 합니다. **진짜 변수는 그게 아닙니다.**
>
> **🔬 하는 일** — 대상 수를 1만에서 5000만까지, 신선도를 24시간에서 30분까지 따로 바꿔 가며 비교합니다.

In [7]:
# ✨ 추가 — 대상 수와 신선도 중 뭐가 더 크게 작용하나
print("① 대상 수만 바꿉니다 (신선도 24시간, 피크 40 QPS 고정)\n")
print(f"{'대상 수':>13}{'배치 $/일':>12}{'온라인 $/일':>13}  싼 쪽")
print("-" * 52)
for n in (10_000, 100_000, 1_000_000, 10_000_000, 50_000_000):
    w = 작업부하("t", n, 300_000, 40, 24.0, 100)
    b, o = 배치안(w, 가격)["돈"], 온라인안(w, 가격)["돈"]
    print(f"{n:>13,}{b:>12.2f}{o:>13.2f}  {'배치' if b < o else '온라인'}")

print("\n\n② 신선도만 바꿉니다 (대상 500만, 피크 120 QPS 고정)\n")
print(f"{'신선도':>10}{'하루 실행':>11}{'배치 $/일':>12}{'온라인 $/일':>13}  싼 쪽")
print("-" * 58)
for h in (24, 12, 6, 4, 3, 2, 1.5, 1.0):
    w = 작업부하("t", 5_000_000, 2_000_000, 120, h, 80)
    안 = 배치안(w, 가격)
    o = 온라인안(w, 가격)["돈"]
    if "이유" in 안:
        print(f"{h:>10}{'불가능':>11}{'—':>12}{o:>13.2f}  온라인 (배치 불가)")
    else:
        회수 = 24.0 / (h - 가격.배치_작업시간)
        print(f"{h:>10}{회수:>11.1f}{안['돈']:>12.2f}{o:>13.2f}  {'배치' if 안['돈'] < o else '온라인'}")

print(f"\n=> ⚠️ 두 표를 비교해 보세요.")
print(f"   ① 대상 수를 5000배(1만 -> 5000만) 늘려도 끝까지 배치가 이깁니다.")
print(f"   ② 신선도를 8배(24시간 -> 3시간) 조였더니 뒤집힙니다.")
print(f"\n   📗 원문은 'Batch cost grows with the number of entities' 라고 합니다. 맞는 말이지만,")
print(f"   실제로 선택을 가르는 건 신선도 요구입니다. 대상 수보다 훨씬 셉니다.")
print(f"\n   왜냐면 신선도를 조이면 하루 실행 횟수가 폭증하기 때문입니다.")
print(f"   24시간이면 하루 1.0회인데 3시간이면 12.0회, 1.5시간이면 48.0회입니다.")
print(f"   대상 수는 아무리 늘려도 '한 번 돌 때 채점할 건수'만 늘릴 뿐입니다.")
print(f"   그리고 온라인은 고정비(복제 2대 = $24/일)라 대상 수와 무관하고요.")
print(f"\n   📝 그래서 설계 회의에서 물어볼 질문은 '사용자가 몇 명인가'보다")
print(f"      '얼마나 최신이어야 하나' 입니다.")

① 대상 수만 바꿉니다 (신선도 24시간, 피크 40 QPS 고정)

         대상 수      배치 $/일      온라인 $/일  싼 쪽
----------------------------------------------------
       10,000        0.08        24.00  배치
      100,000        0.12        24.00  배치
    1,000,000        0.49        24.00  배치
   10,000,000        4.25        24.00  배치
   50,000,000       20.94        24.00  배치


② 신선도만 바꿉니다 (대상 500만, 피크 120 QPS 고정)

       신선도      하루 실행      배치 $/일      온라인 $/일  싼 쪽
----------------------------------------------------------
        24        1.0        2.59        24.00  배치
        12        2.2        4.86        24.00  배치
         6        4.8       10.10        24.00  배치
         4        8.0       16.50        24.00  배치
         3       12.0       24.50        24.00  온라인
         2       24.0       48.50        24.00  온라인
       1.5       48.0       96.50        24.00  온라인
       1.0        불가능           —        24.00  온라인 (배치 불가)

=> ⚠️ 두 표를 비교해 보세요.
   ① 대상 수를 5000배(1만 -> 5000만) 늘려도 끝까지 배치가 이깁니다.
   ② 신선도를

> **👀 주목할 곳** — 두 표의 `싼 쪽` 열. ①은 끝까지 안 바뀌고 ②는 3시간에서 뒤집힙니다.
>
> **✅ 알게 된 사실** — ⚠️ 비용을 가르는 건 대상 수가 아니라 **신선도 요구**입니다.
신선도를 조이면 하루 실행 횟수가 폭증하고(24시간→1.5시간이면 1.0회→48.0회) 비용도 같이 뜁니다.
반면 온라인은 복제 최소 2대라는 **고정비**라 대상 수와 무관합니다.

> **🧩 이 절의 결론**
>
> 서빙 방식은 **가격을 매겨서** 고릅니다. 조건을 만족하는 것 중 가장 싼 것으로요.
> ⚠️ 그리고 선택을 가르는 건 대상 수가 아니라 **신선도 요구**입니다.
>
> **➡️ 다음**
>
> 서빙 방식까지 정했으니 이제 **배포**입니다.
> 한 번에 전부 바꾸면 안 되는데, 그 단계가 6절입니다.

---

## 6. 배포 패턴

<sub>📗 원문 "Deployment patterns"</sub>

새 모델은 **단계를 밟아** 사용자에게 갑니다.

| 단계 | 하는 일 | 보는 것 |
| --- | --- | --- |
| **① 섀도** | 실제 트래픽의 **복사본**을 받습니다. 예측은 기록만 하고 **쓰지 않습니다** | 지연, 오류율, 예측 분포를 기존 모델과 비교 |
| **② 카나리** | **1~5%** 트래픽에 대해 진짜 결정을 합니다. 조금씩 늘립니다 | 가드레일이 깨지면 되돌립니다 |
| **③ A/B 테스트** | 사용자를 무작위로 나눠 각 모델을 보여 줍니다 | **비즈니스 지표**를 통계적으로 비교 |

> 📗 원문: 모든 단계에 **몇 분 안에 되돌릴 수단**이 있어야 합니다.
> 이전 모델을 **배포된 상태로 두고** 설정만 바꿔 트래픽을 돌리세요.

### 🔑 왜 이 순서인가

각 단계가 **직전 단계가 못 보는 것**을 봅니다.

- **섀도**는 "터지지 않나"를 봅니다. 사용자에게 영향이 **0** 이라 가장 안전합니다
- **카나리**는 "실제로 쓰면 어떻게 되나"를 봅니다. 섀도는 결정을 안 하니 못 보던 것이죠
- **A/B**는 "비즈니스가 좋아졌나"를 봅니다. 카나리는 트래픽이 적어 통계적으로 못 봅니다

3절의 지표 세 층과 하나씩 대응합니다.
오프라인 → (섀도) → 온라인 → (카나리) → 비즈니스 → (A/B).

> **🧩 이 절의 결론**
>
> 섀도 → 카나리 → A/B 순서로 갑니다. 각 단계가 **앞 단계가 못 보는 것**을 봅니다.
> 그리고 **몇 분 안에 되돌릴 수 있어야** 합니다.
>
> **➡️ 다음**
>
> 여기까지가 교과서대로입니다.
> 7절은 **잘 알려지지 않은 함정**이고, 측정해 보면 교과서 설명과 좀 다릅니다.

---

## 7. ⚠️ 피드백 루프

<sub>📗 원문 "Feedback loops"</sub>

**모델의 결정이 다음 모델의 학습 데이터를 바꿀 때** 피드백 루프가 생깁니다.

> 📗 Sculley 등의 *Hidden Technical Debt in Machine Learning Systems* 가
> 피드백 루프를 ML 시스템 기술 부채의 주요 원인으로 꼽습니다.

### 📗 흔한 세 경우

| 경우 | 무슨 일이 | 결과 |
| --- | --- | --- |
| **개입이 레이블을 가림** | 이탈 모델이 혜택을 보내고, 혜택이 일부를 붙잡습니다 | 다음 달엔 고위험 고객이 충성 고객처럼 보입니다 |
| **차단된 행동엔 결과가 없음** | 사기 모델이 결제를 막으면 그 건은 **영원히 차지백 레이블이 없습니다** | 막힌 영역의 사기 예시가 점점 줄어듭니다 |
| **노출이 피드백을 제한** | 랭킹 모델이 상위 항목만 보여 줍니다 | 그 항목들만 클릭을 모읍니다 |

### 📗 해법은 하나로 모입니다

**모델이 건드리지 않은 데이터를 남겨 두세요.**

- 작은 무작위 집단에게는 **아무 개입도 안 합니다**
- 일부 트래픽은 **무작위로 결정**합니다
- **점수와 결정을 같이 기록**해서, 나중에 '영향받은 데이터'와 '깨끗한 데이터'를 구분합니다

### ⚠️ 그런데 피해가 어디에 생기는지는 좀 다릅니다

보통 "피드백 루프가 모델을 망가뜨린다"고 설명합니다.
📗 원문도 "retrained model learns that their features do not predict churn" 이라고 하고요.

**재 봤더니 그렇게 단순하지 않았습니다.** 아래에서 직접 확인합니다.

> **실험 7** · ⚠️ 피드백 루프는 무엇을 망가뜨리나
>
> **🎯 배울 것** — 모델이 망가지는 걸까요, **우리가 보는 숫자**가 망가지는 걸까요?
>
> **🔬 하는 일** — 이탈 모델이 상위 N%에 혜택을 보내 일부를 붙잡는 상황을 만들고, ① 재학습한 모델의 성능과 ② 우리가 관측하는 성능을 따로 잽니다.

In [8]:
# ✨ 추가 — 피드백 루프가 모델을 망가뜨리나, 측정을 망가뜨리나
훈련2, y훈련2 = 이탈데이터(4000, 7)
테스트2, y테스트2 = 이탈데이터(2000, 8)
원래모델 = 로지스틱().fit(훈련2, y훈련2)
진짜F1 = F1(y테스트2, 원래모델.predict(테스트2))

훈련점수 = [원래모델._시그모이드(원래모델._벡터(r)) for r in 훈련2]
훈련순위 = sorted(range(len(훈련점수)), key=lambda i: -훈련점수[i])
테스트점수 = [원래모델._시그모이드(원래모델._벡터(r)) for r in 테스트2]
테스트순위 = sorted(range(len(테스트점수)), key=lambda i: -테스트점수[i])

print("상황: 모델이 위험도 상위 N% 에게 혜택을 보내고, 그중 60% 가 실제로 붙잡힙니다")
print(f"      (붙잡히면 레이블이 '이탈 1' -> '유지 0' 으로 바뀝니다)\n")
print(f"개입 전 모델의 진짜 F1 = {진짜F1:.3f}\n")
print(f"{'개입 비율':>9}{'관측 이탈률':>12}{'① 재학습 F1':>13}{'② 관측 F1':>12}{'진짜 F1':>10}")
print("-" * 60)
for 비율 in (0.0, 0.05, 0.10, 0.20, 0.30):
    # ① 훈련 데이터가 오염된 상태로 재학습
    개입훈련 = set(훈련순위[:int(len(훈련순위) * 비율)])
    rng = random.Random(1)
    새훈련y = [0 if (i in 개입훈련 and y훈련2[i] == 1 and rng.random() < 0.6) else y훈련2[i]
             for i in range(len(y훈련2))]
    재학습 = 로지스틱().fit(훈련2, 새훈련y)
    재학습F1 = F1(y테스트2, 재학습.predict(테스트2))

    # ② 테스트 데이터도 오염 — 우리가 '관측'하는 성능
    개입테스트 = set(테스트순위[:int(len(테스트순위) * 비율)])
    rng2 = random.Random(2)
    관측y = [0 if (i in 개입테스트 and y테스트2[i] == 1 and rng2.random() < 0.6) else y테스트2[i]
            for i in range(len(y테스트2))]
    관측F1 = F1(관측y, 원래모델.predict(테스트2))
    print(f"{비율:>9.0%}{sum(관측y)/len(관측y):>12.3f}{재학습F1:>13.3f}"
          f"{관측F1:>12.3f}{진짜F1:>10.3f}")

print(f"\n=> ⚠️ 두 열이 전혀 다르게 움직입니다.")
print(f"\n   ① 재학습 F1 은 거의 안 변합니다. 0.58 근처를 유지하죠.")
print(f"      레이블이 바뀐 건 '모델이 이미 고위험이라고 맞힌' 고객들입니다.")
print(f"      그 사람들의 특징-이탈 관계가 사라져도, 나머지 데이터에 같은 패턴이 남아 있어서")
print(f"      재학습한 모델이 비슷한 계수를 다시 찾아냅니다.")
print(f"\n   ② 관측 F1 은 반토막이 납니다. 0.588 -> 0.276.")
print(f"      모델은 한 글자도 안 바뀌었는데 점수만 떨어집니다.")
print(f"      모델이 '이탈한다'고 한 사람을 혜택이 붙잡았으니, 채점표에는 오답으로 적히거든요.")
print(f"\n   📝 그래서 피드백 루프의 진짜 피해는 이렇게 말하는 게 정확합니다.")
print(f"      모델이 망가지는 게 아니라 '모델을 평가할 능력'이 망가집니다.")
print(f"      그리고 고약하게도 개입이 효과적일수록 모델이 더 못해 보입니다.")
print(f"      성공이 실패처럼 보이는 거죠.")
print(f"\n   💡 그래서 해법이 '깨끗한 데이터를 남겨 두라'인 겁니다.")
print(f"      모델을 지키려는 게 아니라 채점표를 지키려는 것이었습니다.")

상황: 모델이 위험도 상위 N% 에게 혜택을 보내고, 그중 60% 가 실제로 붙잡힙니다
      (붙잡히면 레이블이 '이탈 1' -> '유지 0' 으로 바뀝니다)

개입 전 모델의 진짜 F1 = 0.588

    개입 비율      관측 이탈률     ① 재학습 F1     ② 관측 F1     진짜 F1
------------------------------------------------------------
       0%       0.203        0.588       0.588     0.588
       5%       0.180        0.597       0.501     0.588
      10%       0.161        0.594       0.422     0.588
      20%       0.131        0.577       0.276     0.588
      30%       0.113        0.603       0.276     0.588

=> ⚠️ 두 열이 전혀 다르게 움직입니다.

   ① 재학습 F1 은 거의 안 변합니다. 0.58 근처를 유지하죠.
      레이블이 바뀐 건 '모델이 이미 고위험이라고 맞힌' 고객들입니다.
      그 사람들의 특징-이탈 관계가 사라져도, 나머지 데이터에 같은 패턴이 남아 있어서
      재학습한 모델이 비슷한 계수를 다시 찾아냅니다.

   ② 관측 F1 은 반토막이 납니다. 0.588 -> 0.276.
      모델은 한 글자도 안 바뀌었는데 점수만 떨어집니다.
      모델이 '이탈한다'고 한 사람을 혜택이 붙잡았으니, 채점표에는 오답으로 적히거든요.

   📝 그래서 피드백 루프의 진짜 피해는 이렇게 말하는 게 정확합니다.
      모델이 망가지는 게 아니라 '모델을 평가할 능력'이 망가집니다.
      그리고 고약하게도 개입이 효과적일수록 모델이 더 못해 보입니다.
      성공이 실패처럼 보이는 거죠.

   💡 그래

> **👀 주목할 곳** — `① 재학습 F1` 과 `② 관측 F1` 두 열을 **나란히** 보세요. 전혀 다르게 움직입니다.
>
> **✅ 알게 된 사실** — ⚠️ 재학습 모델은 멀쩡한데 **관측 성능만 0.588 → 0.276** 으로 반토막 납니다.
모델은 한 글자도 안 바뀌었습니다. 혜택이 붙잡은 고객이 채점표에 오답으로 적힐 뿐이죠.
**개입이 효과적일수록 모델이 더 못해 보입니다.** 성공이 실패처럼 보입니다.

> **🧩 이 절의 결론**
>
> ⚠️ 피드백 루프는 **모델이 아니라 채점표를 망가뜨립니다.**
> 그래서 '깨끗한 홀드아웃을 남겨 두라'는 해법이 나온 겁니다. 모델을 지키려는 게 아니라요.
>
> **➡️ 다음**
>
> 설계에서 미리 정해 둘 게 하나 더 있습니다.
> **망가질 때 뭘 할지** 입니다. 8절입니다.

---

## 8. 실패 모드와 안전 기본값

<sub>📗 원문 "Failure modes and safe defaults"</sub>

시스템의 모든 부분은 **언젠가 망가집니다.** 각 실패의 기본 동작을
**설계 때 정해야지, 장애 한가운데서 정하면 안 됩니다.**

| 실패 | 계획이 없으면 | 📗 안전한 기본값 |
| --- | --- | --- |
| 모델 서비스 타임아웃 | 요청이 실패합니다 | **휴리스틱 기준선**을 씁니다 |
| 야간 배치 작업 실패 | 점수가 없거나 낡아집니다 | 정해진 시간까지 **마지막 정상 점수**, 그 뒤엔 규칙 |
| 특징이 없거나 범위를 벗어남 | 모델이 이상한 입력을 채점합니다 | **문서에 적힌 기본값**을 쓰고 기록을 남깁니다 |
| 이력 없는 신규 대상 | 점수가 없습니다 | 규칙이나 전체 평균 |
| 점수 분포가 갑자기 이동 | 대규모로 잘못된 결정 | **자동 행동을 멈추고** 사람을 호출합니다 |

### 🔑 3절의 규칙이 여기서 다시 나옵니다

표의 **기본값 열에 '휴리스틱 기준선'이 두 번** 나옵니다.

> 📗 원문: "The heuristic baseline from your metric section is also the default
> for most failures, so **build it first.**"

3절에서 비교용으로 만든 규칙이 **장애 시 대체 시스템**이 됩니다.
그래서 규칙을 먼저 만들라는 거예요. 두 번 쓰입니다.

> **🧩 이 절의 결론**
>
> 실패마다 **기본 동작을 설계 때 정합니다.** 장애 중에 정하면 늦습니다.
> 그리고 3절의 **휴리스틱 규칙이 대부분의 기본값**이 됩니다. 그래서 먼저 만듭니다.
>
> **➡️ 다음**
>
> 여기까지 정한 것들을 **문서 한 장**으로 묶습니다.
> 그리고 빠진 게 없는지 자동으로 채점합니다. 9절입니다.

---

## 9. 설계 문서 채점기

<sub>📗 원문 "The design document"</sub>

지금까지의 결정을 **학습 시작 전에** 한 문서에 적습니다.
그리고 체크리스트로 채점합니다.

📗 원문은 **여섯 항목을 블로킹**으로 둡니다. 하나라도 빠지면 시스템이
**위험하거나 쓸모없어지기** 때문입니다.

| 항목 | 묻는 것 | 어느 절 |
| --- | --- | --- |
| **decision** | 각 예측을 쓰는 행동이 뭔가 | 1절 |
| **business_metric** | 모델이 움직여야 할 비즈니스 지표 | 3절 |
| **baselines** | 휴리스틱 또는 현재 시스템 기준선 | 3절 |
| **no_leakage** | 모든 특징이 예측 시점에 존재하나 | 📗 데이터 요구사항 |
| **rollout** | 섀도/카나리 단계와 롤백 | 6절 |
| **failure_modes** | 모든 실패에 기본 동작이 있나 | 8절 |

> **실험 9** · 설계 문서 채점기
>
> **🎯 배울 것** — 빠진 항목을 **사람이 아니라 코드가** 찾아냅니다.
>
> **🔬 하는 일** — 체크리스트를 코드로 만들고, 부실한 문서와 완성된 문서를 각각 채점해 봅니다.

In [9]:
# 📗 원문의 CheckItem / review_design_doc 구조를 옮깁니다
@dataclass(frozen=True)
class 점검항목:
    키: str
    가중치: int
    블로킹: bool
    메시지: str
    검사: object


def 경로값(문서, 경로):
    현재 = 문서
    for 조각 in 경로.split("."):
        if not isinstance(현재, dict) or 조각 not in 현재:
            return None
        현재 = 현재[조각]
    return 현재


def 글있나(문서, 경로):
    v = 경로값(문서, 경로)
    return isinstance(v, str) and len(v.strip()) > 0


def 목록있나(문서, 경로):
    v = 경로값(문서, 경로)
    return isinstance(v, list) and len(v) > 0


def 기준선검사(문서):
    목록 = 경로값(문서, "baselines") or []
    종류 = {b.get("kind") for b in 목록 if isinstance(b, dict)}
    return bool(종류 & {"heuristic", "current_system"})      # 자명한 기준선만으론 부족


def 실패모드검사(문서):
    목록 = 경로값(문서, "failure_modes") or []
    return bool(목록) and all(isinstance(f, dict) and f.get("default") for f in 목록)


점검표 = [
    점검항목("decision", 2, True, "각 예측을 쓰는 행동을 적으세요",
          lambda d: 글있나(d, "problem.decision")),
    점검항목("business_metric", 2, True, "모델이 움직여야 할 비즈니스 지표를 적으세요",
          lambda d: 글있나(d, "metrics.business")),
    점검항목("baselines", 2, True, "휴리스틱 또는 현재 시스템 기준선을 넣으세요", 기준선검사),
    점검항목("no_leakage", 2, True, "모든 특징이 예측 시점에 존재함을 확인하세요",
          lambda d: 경로값(d, "data.features_available_at_prediction_time") is True),
    점검항목("rollout", 2, True, "섀도/카나리 단계와 롤백을 계획하세요",
          lambda d: 목록있나(d, "rollout.stages") and 글있나(d, "rollout.rollback")),
    점검항목("failure_modes", 2, True, "모든 실패 모드에 기본 동작을 주세요", 실패모드검사),
    점검항목("online_metric", 1, False, "온라인 지표를 적으세요",
          lambda d: 목록있나(d, "metrics.online")),
    점검항목("guardrails", 1, False, "나빠지면 안 되는 가드레일 지표를 적으세요",
          lambda d: 목록있나(d, "metrics.guardrails")),
    점검항목("budgets", 1, False, "지연·피크 QPS·일일 비용 예산을 정하세요",
          lambda d: all(경로값(d, f"budgets.{k}") is not None
                        for k in ("latency_ms", "peak_qps", "dollars_per_day"))),
    점검항목("label_delay", 1, False, "레이블 지연을 일 단위로 적으세요",
          lambda d: 경로값(d, "data.label_delay_days") is not None),
    점검항목("feedback_loop", 1, False, "피드백 루프 위험과 완화책을 적으세요",
          lambda d: 글있나(d, "feedback_loop.risk") and 글있나(d, "feedback_loop.mitigation")),
    점검항목("monitoring", 1, False, "출시 후 무엇을 볼지 적으세요",
          lambda d: 목록있나(d, "monitoring")),
]


def 문서채점(문서):
    통과, 실패 = [], []
    총점 = 얻은점 = 0
    블로킹실패 = False
    for 항 in 점검표:
        총점 += 항.가중치
        try:
            결과 = bool(항.검사(문서))
        except Exception:
            결과 = False
        if 결과:
            얻은점 += 항.가중치
            통과.append(항)
        else:
            실패.append(항)
            if 항.블로킹:
                블로킹실패 = True
    점수 = 얻은점 / 총점
    판정 = "검토 준비됨" if (점수 >= 0.85 and not 블로킹실패) else "수정 필요"
    return 점수, 판정, 실패


부실한문서 = {
    "problem": {"prediction": "이 사용자가 해지할지"},
    "metrics": {"offline": ["AUC"]},
    "baselines": [{"kind": "trivial", "name": "다수 클래스"}],
    "data": {"label_source": "subscriptions 테이블"},
}

완성된문서 = {
    "problem": {"prediction": "30일 안에 해지할 확률",
                "decision": "매일 밤 상위 5%에게 혜택 쿠폰 발송",
                "why_not_rules": "규칙이 12개로 늘어나 유지가 어려움"},
    "metrics": {"offline": ["상위 5% 정밀도"], "online": ["혜택 수락률"],
                "business": "처치군 30일 이탈률",
                "guardrails": ["구한 고객당 비용", "수신거부율", "p99 지연"]},
    "baselines": [{"kind": "trivial", "name": "다수 클래스"},
                  {"kind": "heuristic", "name": "days_idle > 14"},
                  {"kind": "current_system", "name": "지원팀 수동 목록"}],
    "data": {"label_source": "subscriptions.canceled_at", "label_delay_days": 30,
             "features_available_at_prediction_time": True},
    "budgets": {"latency_ms": 60000, "peak_qps": 40, "dollars_per_day": 10,
                "max_staleness_hours": 24},
    "serving": {"mode": "batch"},
    "rollout": {"stages": ["shadow", "canary 5%"], "rollback": "설정 플래그로 즉시 전환"},
    "feedback_loop": {"risk": "혜택이 고위험 고객을 붙잡아 레이블을 가림",
                      "mitigation": "무작위 2%는 혜택 제외 (깨끗한 홀드아웃)"},
    "failure_modes": [{"failure": "모델 타임아웃", "default": "규칙 사용"},
                      {"failure": "배치 실패", "default": "마지막 정상 점수 24시간 유지"}],
    "monitoring": ["점수 분포", "일일 발송 건수", "특징 결측률"],
}

for 이름, 문서 in [("부실한 초안", 부실한문서), ("완성된 문서", 완성된문서)]:
    점수, 판정, 실패 = 문서채점(문서)
    print(f"\n{이름}: 점수 {점수:.2f}, 판정: {판정}")
    for 항 in 실패:
        꼬리 = "BLOCKING" if 항.블로킹 else "미작성  "
        print(f"   {꼬리} {항.키:<18} {항.메시지}")
    if not 실패:
        print("   빠진 항목 없음")

print(f"\n=> 부실한 초안은 블로킹만 여럿입니다. AUC 0.91 은 아무 데도 안 나옵니다.")
print(f"   도입부의 실패 사례가 정확히 이 상태였습니다. 모델은 좋은데 설계가 비어 있었죠.")


부실한 초안: 점수 0.00, 판정: 수정 필요
   BLOCKING decision           각 예측을 쓰는 행동을 적으세요
   BLOCKING business_metric    모델이 움직여야 할 비즈니스 지표를 적으세요
   BLOCKING baselines          휴리스틱 또는 현재 시스템 기준선을 넣으세요
   BLOCKING no_leakage         모든 특징이 예측 시점에 존재함을 확인하세요
   BLOCKING rollout            섀도/카나리 단계와 롤백을 계획하세요
   BLOCKING failure_modes      모든 실패 모드에 기본 동작을 주세요
   미작성   online_metric      온라인 지표를 적으세요
   미작성   guardrails         나빠지면 안 되는 가드레일 지표를 적으세요
   미작성   budgets            지연·피크 QPS·일일 비용 예산을 정하세요
   미작성   label_delay        레이블 지연을 일 단위로 적으세요
   미작성   feedback_loop      피드백 루프 위험과 완화책을 적으세요
   미작성   monitoring         출시 후 무엇을 볼지 적으세요

완성된 문서: 점수 1.00, 판정: 검토 준비됨
   빠진 항목 없음

=> 부실한 초안은 블로킹만 여럿입니다. AUC 0.91 은 아무 데도 안 나옵니다.
   도입부의 실패 사례가 정확히 이 상태였습니다. 모델은 좋은데 설계가 비어 있었죠.


> **👀 주목할 곳** — 부실한 초안의 `BLOCKING` 줄들. 그리고 완성된 문서의 점수.
>
> **✅ 알게 된 사실** — 블로킹 항목은 **모델 성능과 아무 상관이 없습니다.** 전부 설계 질문이죠.
도입부의 실패 사례(AUC 0.91인데 아무도 안 씀)가 정확히 이 상태였습니다.

> **🧩 이 절의 결론**
>
> 설계 문서는 **학습 전에** 씁니다. 그리고 체크리스트로 채점합니다.
> 블로킹 여섯 개는 전부 **모델 성능과 무관한 설계 질문**입니다.
>
> **➡️ 다음**
>
> 10절에서 전체를 정리하고 Ship It 스킬로 마칩니다.
> 그리고 **20단원**에서 배포한 모델을 어떻게 감시할지 봅니다.

---

## 10. 정리

### ⚠️ 이 노트북에서 원문과 달랐던 것

| 📗 원문의 설명 | 측정 결과 | 절 |
| --- | --- | --- |
| "배치 비용은 **대상 수**에 비례해 커진다" | 맞지만 **선택을 가르는 건 신선도**입니다. 대상 수를 5000배 늘려도 배치가 이기는데, 신선도를 8배 조이면 뒤집힙니다 | 5 |
| 피드백 루프 — "재학습 모델이 그 특징은 이탈을 예측하지 않는다고 배운다" | ⚠️ **재학습 모델은 거의 안 변합니다**(F1 0.588 유지). 망가지는 건 **채점표**입니다. 관측 F1이 0.588 → 0.276 | 7 |

**측정으로 확인된** 것들입니다.

| 📗 원문의 주장 | 결과 | 절 |
| --- | --- | --- |
| 자명한 기준선은 약하고 휴리스틱이 진짜 상대 | ✅ 다수 클래스 F1 0.000, 규칙 0.529 | 3 |
| 특징이 약하면 규칙이 모델을 이긴다 | ✅ tickets·tenure 영향을 줄이니 규칙이 역전 | 3 |
| Little's 법칙 (400 QPS × 50ms = 20건) | ✅ 그대로 | 4 |
| 배치 작업보다 짧은 신선도는 불가능 | ✅ 3분 신선도에서 배치가 아예 탈락 | 5 |
| 📗 원문 출력 (배치 $0.91, 온라인 $24.00) | ✅ 재현 | 5 |

### 🔑 한 줄로 꿰면

> **"무슨 결정을 하나"가 신선도를 정하고, 신선도가 서빙 방식을 정하고,
> 서빙 방식이 비용을 정합니다.** 그 사슬의 맨 앞이 1절의 질문입니다.

그리고 **휴리스틱 규칙을 가장 먼저 만드세요.** 두 번 쓰입니다.
3절에서는 비교 대상으로, 8절에서는 장애 시 대체 시스템으로요.

---

## 🚀 Ship It — `skill-ml-design-review.md`

<sub>📗 원문 저장소의 `19-ml-system-design/outputs/skill-ml-design-review.md` (84줄)</sub>

> 📗 원문: 이 스킬은 ML 설계 문서를 절별로 검토하고 **블로킹 공백을 먼저** 짚습니다.
> 그리고 이 단원의 배치·온라인 비용 계산으로 서빙 선택을 확인합니다.

### 검토 순서 (블로킹 먼저)

| 순서 | 묻는 것 | 통과 기준 |
| --- | --- | --- |
| 1 | 각 예측을 쓰는 **행동**이 적혀 있나 | 누가 무엇을 하는지 한 문장 |
| 2 | **비즈니스 지표**가 있나 | 실험으로 잴 수 있는 것 |
| 3 | **휴리스틱/현재 시스템** 기준선이 있나 | 자명한 기준선만으론 불충분 |
| 4 | 모든 특징이 **예측 시점에 존재**하나 | 누출 확인 |
| 5 | **롤아웃 단계와 롤백**이 있나 | 섀도 또는 카나리 + 되돌리는 법 |
| 6 | 모든 실패에 **기본 동작**이 있나 | 빈칸이 없어야 함 |

### ✨ 추가 — 측정 뒤에 제가 덧붙인 두 질문

```markdown
## 서빙 선택을 검토할 때 (⚠️ 5절)

"사용자가 몇 명인가"보다 **"얼마나 최신이어야 하나"** 를 먼저 물으세요.
대상 수는 5000배 차이가 나도 선택이 안 바뀌는데,
신선도는 8배만 조여도 배치에서 온라인으로 뒤집힙니다.

그리고 **배치 작업 1회 소요 시간**을 확인하세요.
그보다 짧은 신선도는 어떤 예산으로도 불가능합니다.

## 피드백 루프를 검토할 때 (⚠️ 7절)

"모델이 망가질까요?" 가 아니라
**"모델을 계속 평가할 수 있을까요?"** 를 물으세요.

개입이 있으면 관측 성능이 실제보다 나쁘게 나옵니다.
그리고 개입이 효과적일수록 더 나쁘게 나옵니다.
깨끗한 홀드아웃이 없으면 **성공을 실패로 오해**하고 모델을 내리게 됩니다.
```

---

## 📚 용어 정리

<sub>📗 원문 "Key Terms" — 설명은 쉬운 말로 다시 썼습니다</sub>

| 용어 | 흔히 하는 말 | 진짜 무슨 뜻인가 |
| --- | --- | --- |
| **ML 과제** | "이탈 모델 만들자" | 예측 대상 + 엔티티 + 기간 + **그 예측을 쓰는 결정** |
| **오프라인 지표** | "모델 점수" | 출시 전 과거 홀드아웃 데이터로 잰 지표 |
| **온라인 지표** | "실시간 수치" | 모델이 돌아가는 동안 실제 트래픽에서 잰 지표 |
| **비즈니스 지표** | "목표" | 결정이 바꿔야 할 회사 성과. 실험으로 잽니다 |
| **가드레일 지표** | "해치지는 말자" | 새 모델이 나빠뜨리면 안 되는 지표 |
| **기준선** | "멍청한 모델" | 모델이 정해진 폭만큼 이겨야 하는 상대. 자명·휴리스틱·현재 시스템 |
| **배치 추론** | "미리 계산한 점수" | 정해진 작업이 전부 채점해 저장해 두고 조회만 |
| **온라인 추론** | "실시간 모델" | 요청이 올 때 그 자리에서 채점 |
| **섀도 배포** | "다크 런치" | 새 모델이 실 트래픽을 채점하되 **아무도 그 결과를 안 씀** |
| **카나리** | "점진 배포" | 작게 시작해 비율을 늘리는 실제 배포. 롤백 준비 필수 |
| **피드백 루프** | "모델이 자기를 학습한다" | 모델의 결정이 다음 버전의 학습 데이터를 바꾸는 것 |
| **p99 지연** | — | 요청 100건 중 99건이 그 안에 끝나는 시간. 평균이 아닙니다 |

---

## 📖 더 읽을 것

<sub>📗 원문 "Further Reading"</sub>

- [Sculley et al., *Hidden Technical Debt in Machine Learning Systems* (NeurIPS 2015)](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html) — 7절 피드백 루프의 원 논문
- [Google, *Rules of Machine Learning*](https://developers.google.com/machine-learning/guides/rules-of-ml) — 2절의 1번·3번 규칙
- [Breck et al., *The ML Test Score* (IEEE Big Data 2017)](https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/) — 28개 항목짜리 production 준비도 평가표
- [Kohavi, Tang, Xu, *Trustworthy Online Controlled Experiments* (2020)](https://experimentguide.com/) — 6절 A/B 테스트를 제대로 하는 법
- [Google Cloud, *MLOps*](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) — 학습·서빙 자동화의 세 성숙 단계

---

## ✅ 이해 확인 퀴즈

<sub>📗 원문 저장소의 `19-ml-system-design/quiz.json` + ✨ 측정 결과를 묻는 문항</sub>

**1. "이탈 모델을 만들자"가 아직 ML 과제가 아닌 이유는?**
1. 알고리즘을 안 정해서
2. **대상·기간·그 예측을 쓸 결정이 안 정해져서**
3. 데이터가 없어서
4. 지표를 안 정해서

**2. 모델 F1이 0.56입니다. 가장 먼저 확인할 것은?**
1. 하이퍼파라미터를 더 튜닝한다
2. **기존에 쓰던 규칙의 F1을 잰다**
3. 더 복잡한 모델로 바꾼다
4. 특징을 더 넣는다

**3. 신선도 요구가 3분인데 배치 작업이 1시간 걸립니다.**
1. 서버를 늘려 배치를 빠르게 한다
2. **배치는 불가능합니다. 온라인이어야 합니다**
3. 신선도를 포기한다
4. 비용을 더 쓰면 됩니다

**4. ⚠️ 배치와 온라인 중 선택을 가장 크게 가르는 것은?**
1. 대상(엔티티) 수
2. **신선도 요구**
3. 모델 크기
4. 특징 개수

**5. ⚠️ 피드백 루프가 망가뜨리는 것은?**
1. 재학습한 모델의 성능
2. **모델을 평가하는 능력 (관측 지표)**
3. 학습 속도
4. 예측 지연

In [10]:
문항들 = [
    ("ML 과제의 조건", 2,
     "예측 대상과 기간이 안 정해졌고, 무엇보다 그 숫자를 보고 누가 뭘 할지가 없습니다. "
     "결정이 정해져야 신선도가 정해지고, 신선도가 서빙 방식을 정합니다.",
     "1절 (같은 예측, 다른 결정 -> 다른 시스템)"),
    ("F1 0.56 을 받았을 때", 2,
     "0.56 은 혼자서는 아무 뜻이 없습니다. 측정 결과 다수 클래스는 0.000, 규칙은 0.529 였습니다. "
     "리프트는 가장 센 기준선 기준으로 재야 합니다.",
     "3절 (기준선 비교와 규칙이 이기는 경우)"),
    ("3분 신선도 + 1시간 배치", 2,
     "배치 작업이 끝나기도 전에 다음 갱신 시각이 옵니다. 가격 문제가 아니라 구조적으로 불가능합니다. "
     "5절 코드가 이 경우를 '불가'로 걸러 냅니다.",
     "1절 (요구사항 비교) · 5절 (가격 계산)"),
    ("선택을 가르는 변수", 2,
     "대상 수를 1만에서 5000만으로(5000배) 늘려도 배치가 끝까지 이깁니다. "
     "반면 신선도를 24시간에서 3시간으로(8배) 조이면 뒤집힙니다. "
     "신선도를 조이면 하루 실행 횟수가 1.0회에서 12.0회로 폭증하기 때문입니다.",
     "5절 (대상 수와 신선도를 따로 바꿔 측정)"),
    ("피드백 루프의 진짜 피해", 2,
     "재학습 모델의 F1 은 0.588 근처를 유지합니다. 그런데 관측 F1 은 0.588 -> 0.276 으로 반토막 납니다. "
     "모델은 안 바뀌고 채점표가 틀려지는 겁니다. 게다가 개입이 효과적일수록 더 나쁘게 보입니다.",
     "7절 (재학습 성능과 관측 성능을 따로 측정)"),
]

print("=" * 80)
for 번호, (제목, 정답, 해설, 어디) in enumerate(문항들, 1):
    print(f"{번호}. {제목}")
    print(f"   정답: {정답}번")
    print(f"   해설: {해설}")
    print(f"   이 노트북: {어디}")
    print("-" * 80)

print("\n채점 기준")
print("   5개 다 맞음 -> 20단원(모니터링과 드리프트)으로 넘어가세요")
print("   3~4개 맞음  -> 틀린 문항의 '이 노트북' 절만 다시 읽으세요")
print("   2개 이하    -> 1절(결정이 모든 걸 끌고 간다)과 3절(기준선)을 먼저 보세요")

print("\n\n19~21단원이 한 묶음입니다")
print("   19단원  설계    — 배포 전에 정할 것")
print("   20단원  감시    — 배포한 뒤 무엇을 볼 것인가")
print("   21단원  일관성  — 학습 때와 서빙 때 특징이 달라지는 문제")
print("\n   셋 다 '모델 바깥' 이야기입니다. 1~18단원이 모델 안쪽이었다면요.")

1. ML 과제의 조건
   정답: 2번
   해설: 예측 대상과 기간이 안 정해졌고, 무엇보다 그 숫자를 보고 누가 뭘 할지가 없습니다. 결정이 정해져야 신선도가 정해지고, 신선도가 서빙 방식을 정합니다.
   이 노트북: 1절 (같은 예측, 다른 결정 -> 다른 시스템)
--------------------------------------------------------------------------------
2. F1 0.56 을 받았을 때
   정답: 2번
   해설: 0.56 은 혼자서는 아무 뜻이 없습니다. 측정 결과 다수 클래스는 0.000, 규칙은 0.529 였습니다. 리프트는 가장 센 기준선 기준으로 재야 합니다.
   이 노트북: 3절 (기준선 비교와 규칙이 이기는 경우)
--------------------------------------------------------------------------------
3. 3분 신선도 + 1시간 배치
   정답: 2번
   해설: 배치 작업이 끝나기도 전에 다음 갱신 시각이 옵니다. 가격 문제가 아니라 구조적으로 불가능합니다. 5절 코드가 이 경우를 '불가'로 걸러 냅니다.
   이 노트북: 1절 (요구사항 비교) · 5절 (가격 계산)
--------------------------------------------------------------------------------
4. 선택을 가르는 변수
   정답: 2번
   해설: 대상 수를 1만에서 5000만으로(5000배) 늘려도 배치가 끝까지 이깁니다. 반면 신선도를 24시간에서 3시간으로(8배) 조이면 뒤집힙니다. 신선도를 조이면 하루 실행 횟수가 1.0회에서 12.0회로 폭증하기 때문입니다.
   이 노트북: 5절 (대상 수와 신선도를 따로 바꿔 측정)
--------------------------------------------------------------------------------
5. 피드백 루프의 진짜 피해
  